# Cobertura da Base de Dados

Valida se todas as empresas que deveriam estar no S&P 500 em cada semestre realmente têm dados de preço na base construída.

| Parte | O que faz |
|---|---|
| 1 | Carrega base_completa e composição histórica |
| 2 | Cobertura por semestre — extensão 2016/S1–2025/S2 |
| 3 | Tickers cronicamente ausentes na extensão |
| 4 | Cobertura de referência — base do professor 1996–2015 |
| 5 | Resumo final |

In [58]:
import pandas as pd
from pathlib import Path

DATA_DIR     = Path("../data_bases")
EXTERNAL_DIR = DATA_DIR / "external"

META_COLS = {"year", "semester", "period", "day_in_semester", "total_days_sem"}

print("Pronto.")

Pronto.


---
## Parte 1 — Carregar dados

In [59]:
# Base completa (professor + extensão)
base = pd.read_csv(DATA_DIR / "base_completa.csv", index_col=0, parse_dates=True)

# Separar base do professor e extensão
prof = base[base["year"] <= 2015]
ext  = base[base["year"] >= 2016]

ticker_cols = [c for c in base.columns if c not in META_COLS]

print(f"Base completa: {base.shape[0]} dias × {len(ticker_cols)} tickers")
print(f"  Professor (1990–2015): {len(prof)} dias")
print(f"  Extensão  (2016–2025): {len(ext)} dias")

# Composição histórica do S&P 500
hist_sp500 = pd.read_csv(EXTERNAL_DIR / "sp500_historico.csv")
hist_sp500["date"] = pd.to_datetime(hist_sp500["date"])

def get_constituents(date_str):
    date   = pd.to_datetime(date_str)
    subset = hist_sp500[hist_sp500["date"] <= date]
    if len(subset) == 0:
        return set()
    return set(subset.iloc[-1]["tickers"].split(","))

print(f"\nComposição: {len(hist_sp500)} snapshots ({hist_sp500['date'].min().date()} → {hist_sp500['date'].max().date()})")

Base completa: 8939 dias × 1343 tickers
  Professor (1990–2015): 6425 dias
  Extensão  (2016–2025): 2514 dias

Composição: 2705 snapshots (1996-01-02 → 2026-01-14)


---
## Parte 2 — Cobertura por semestre (extensão 2016–2025)

Para cada semestre, comparamos os tickers esperados (composição do S&P 500) com os tickers que realmente têm pelo menos um preço não-nulo na base.

In [60]:
# Semestres da extensão
semesters_ext = []
for year in range(2016, 2026):
    for sem in [1, 2]:
        fim = f"{year}-06-30" if sem == 1 else f"{year}-12-31"
        semesters_ext.append({"period": f"{year}/S{sem}", "year": year, "semester": sem, "fim": fim})

rows_cov = []
for s in semesters_ext:
    mask     = ext["period"] == s["period"]
    sem_data = ext.loc[mask, ticker_cols]

    if len(sem_data) == 0:
        continue

    expected       = get_constituents(s["fim"])
    expected_in_df = expected & set(ticker_cols)   # tickers que são colunas na base
    not_in_df      = expected - set(ticker_cols)   # esperados mas nem como coluna

    has_data = {t for t in expected_in_df if sem_data[t].notna().any()}
    missing  = (expected_in_df - has_data) | not_in_df

    rows_cov.append({
        "period":       s["period"],
        "n_dias":       len(sem_data),
        "n_expected":   len(expected),
        "n_present":    len(has_data),
        "n_missing":    len(missing),
        "coverage_pct": round(len(has_data) / len(expected) * 100, 1) if expected else 100.0,
        "missing_list": sorted(missing),
    })

df_cov = pd.DataFrame(rows_cov)
print(df_cov[["period","n_dias","n_expected","n_present","n_missing","coverage_pct"]].to_string(index=False))
print(f"\nCobertura média: {df_cov['coverage_pct'].mean():.1f}%")
print(f"Pior semestre:   {df_cov.loc[df_cov['coverage_pct'].idxmin(), 'period']} ({df_cov['coverage_pct'].min():.1f}%)")
print(f"Melhor semestre: {df_cov.loc[df_cov['coverage_pct'].idxmax(), 'period']} ({df_cov['coverage_pct'].max():.1f}%)")

 period  n_dias  n_expected  n_present  n_missing  coverage_pct
2016/S1     125         505        505          0         100.0
2016/S2     127         506        506          0         100.0
2017/S1     125         506        506          0         100.0
2017/S2     126         505        505          0         100.0
2018/S1     125         506        506          0         100.0
2018/S2     126         505        505          0         100.0
2019/S1     124         505        505          0         100.0
2019/S2     128         505        505          0         100.0
2020/S1     125         505        505          0         100.0
2020/S2     128         505        505          0         100.0
2021/S1     124         505        505          0         100.0
2021/S2     128         505        505          0         100.0
2022/S1     124         503        503          0         100.0
2022/S2     127         503        503          0         100.0
2023/S1     124         503        503  

---
## Parte 3 — Tickers cronicamente ausentes na extensão

Identifica tickers que estavam no S&P 500 durante a extensão mas nunca têm dados.

In [61]:
from collections import defaultdict

ausenteS = defaultdict(list)
for row in rows_cov:
    for t in row["missing_list"]:
        ausenteS[t].append(row["period"])

rows_aus = []
for ticker, periods_faltou in sorted(ausenteS.items()):
    rows_aus.append({
        "ticker":        ticker,
        "n_sem_ausente": len(periods_faltou),
        "primeiro_sem":  periods_faltou[0],
        "ultimo_sem":    periods_faltou[-1],
    })

if not rows_aus:
    print("✅ Cobertura 100% — nenhum ticker ausente na extensão.")
    df_aus = pd.DataFrame(columns=["ticker", "n_sem_ausente", "primeiro_sem", "ultimo_sem"])
else:
    df_aus = pd.DataFrame(rows_aus).sort_values("n_sem_ausente", ascending=False)
    print(f"Tickers com dados faltantes: {len(df_aus)}")
    print()
    print(df_aus.to_string(index=False))

✅ Cobertura 100% — nenhum ticker ausente na extensão.


---
## Parte 4 — Cobertura de referência (base do professor 1996–2015)

O `sp500_historico.csv` começa em 1996. Verificamos os semestres com overlap (1996/S1–2015/S2) como referência — a base do professor é considerada correta, mas queremos identificar discrepâncias óbvias.

In [62]:
ticker_cols_prof = [c for c in prof.columns if c not in META_COLS]

# Semestres do professor que têm cobertura no sp500_historico (a partir de 1996)
semesters_prof = []
for year in range(1996, 2016):
    for sem in [1, 2]:
        fim = f"{year}-06-30" if sem == 1 else f"{year}-12-31"
        semesters_prof.append({"period": f"{year}/S{sem}", "year": year, "semester": sem, "fim": fim})

rows_prof = []
for s in semesters_prof:
    mask     = prof["period"] == s["period"]
    sem_data = prof.loc[mask, ticker_cols_prof]
    if len(sem_data) == 0:
        continue

    expected       = get_constituents(s["fim"])
    expected_in_df = expected & set(ticker_cols_prof)
    not_in_df      = expected - set(ticker_cols_prof)

    has_data = {t for t in expected_in_df if sem_data[t].notna().any()}
    missing  = (expected_in_df - has_data) | not_in_df

    rows_prof.append({
        "period":       s["period"],
        "n_expected":   len(expected),
        "n_present":    len(has_data),
        "n_missing":    len(missing),
        "coverage_pct": round(len(has_data) / len(expected) * 100, 1) if expected else 100.0,
    })

df_prof_cov = pd.DataFrame(rows_prof)
print("Cobertura da base do professor vs sp500_historico (referência):")
print(df_prof_cov.to_string(index=False))
print(f"\nCobertura média: {df_prof_cov['coverage_pct'].mean():.1f}%")
print("\nNota: discrepâncias aqui são normais — composições podem diferir entre fontes.")

Cobertura da base do professor vs sp500_historico (referência):
 period  n_expected  n_present  n_missing  coverage_pct
1996/S1         487        342        145          70.2
1996/S2         488        344        144          70.5
1997/S1         488        352        136          72.1
1997/S2         488        360        128          73.8
1998/S1         490        367        123          74.9
1998/S2         493        381        112          77.3
1999/S1         491        386        105          78.6
1999/S2         492        401         91          81.5
2000/S1         492        407         85          82.7
2000/S2         492        413         79          83.9
2001/S1         495        419         76          84.6
2001/S2         496        424         72          85.5
2002/S1         495        430         65          86.9
2002/S2         494        433         61          87.7
2003/S1         494        436         58          88.3
2003/S2         494        437         5

---
## Parte 5 — Resumo final

---
## Parte 6 — Cruzamento: tickers_ausentes vs coleta_report_validacao

In [63]:
aus    = pd.read_csv(EXTERNAL_DIR / "tickers_ausentes.csv")
coleta = pd.read_csv(EXTERNAL_DIR / "coleta_report_validacao.csv")

# Verificar quais tickers ok_tiingo realmente têm coluna COM dados na extensão atual
# (checa o CSV em disco, não o arquivo tickers_ausentes que pode estar desatualizado)
ext_check = pd.read_csv(DATA_DIR / "extensao_2016_2025.csv", index_col=0)
tickers_na_base = set(ext_check.columns) - META_COLS
tickers_com_dado = {
    c for c in tickers_na_base
    if ext_check[c].notna().any()
}

set_aus    = set(aus["ticker"])
set_coleta = set(coleta["ticker"])

STATUS_FAIL    = {"irrecuperavel", "deletado_ticker_reciclado"}
# ok_tiingo vai para [P] se o ticker já tem dado na extensão; [⚠] só se ainda está faltando
STATUS_PARTIAL = {"lacuna_legitima", "parcial_sem_21CF", "parcial_sem_2016"}

set_irrecuperavel = set(coleta.loc[coleta["status"].isin(STATUS_FAIL), "ticker"])
set_partial       = set(coleta.loc[coleta["status"].isin(STATUS_PARTIAL), "ticker"])
set_ok_tiingo     = set(coleta.loc[coleta["status"] == "ok_tiingo", "ticker"])

# [⚠] só se ok_tiingo E ainda ausente da extensão
grupo_re_run  = set_aus & set_ok_tiingo - tickers_com_dado
# [P] ok_tiingo já incorporado OU parcial/legítima
grupo_parcial = (set_aus & set_partial) | (set_aus & set_ok_tiingo & tickers_com_dado)
grupo_irrec   = set_aus & set_irrecuperavel
so_ausente    = set_aus - set_coleta
so_coleta     = set_irrecuperavel - set_aus

PRICES_YAHOO  = DATA_DIR / "prices"
PRICES_TIINGO = DATA_DIR / "prices_tiingo"

print("=" * 65)
print(f"Tickers ausentes na base:          {len(set_aus):>3}")
print(f"  Irrecuperáveis:                  {len(grupo_irrec):>3}")
print(f"  Ausências esperadas/parciais:    {len(grupo_parcial):>3}")
print(f"  Novo dado — pipeline pendente:   {len(grupo_re_run):>3}")
print("=" * 65)

if grupo_re_run:
    print(f"\n[⚠] Dado NOVO — re-rodar pipeline Parts 4→6→7 ({len(grupo_re_run)}):")
    for t in sorted(grupo_re_run):
        aus_row = aus[aus["ticker"] == t].iloc[0]
        fonte = "tiingo" if (PRICES_TIINGO / f"{t}.csv").exists() else "yahoo"
        print(f"  {t:8s}  fonte={fonte}  {aus_row['n_sem_ausente']} sem ausentes")

if grupo_parcial:
    print(f"\n[P] Ausências esperadas ou parciais ({len(grupo_parcial)}) — já tratadas na base:")
    for t in sorted(grupo_parcial):
        row     = coleta[coleta["ticker"] == t].iloc[0]
        aus_row = aus[aus["ticker"] == t].iloc[0]
        ja = t in tickers_com_dado
        print(f"  {t:8s}  status={row['status']:<25s}  {aus_row['n_sem_ausente']} sem  {'(dados na base)' if ja else '(sem dado)'}")

print(f"\n[A] Irrecuperáveis ({len(grupo_irrec)}) — sem dados em nenhuma fonte:")
for t in sorted(grupo_irrec):
    row     = coleta[coleta["ticker"] == t].iloc[0]
    aus_row = aus[aus["ticker"] == t].iloc[0]
    print(f"  {t:8s}  status={row['status']:<35s}  {aus_row['n_sem_ausente']} semestres")

if so_ausente:
    print(f"\n[B] Só ausente ({len(so_ausente)}) — arquivo existe mas dados não chegam:")
    for t in sorted(so_ausente):
        aus_row = aus[aus["ticker"] == t].iloc[0]
        print(f"  {t:8s}  {aus_row['n_sem_ausente']} sem")

if so_coleta:
    print(f"\n[C] Só na coleta ({len(so_coleta)}) — falhou mas base tem dados:")
    for t in sorted(so_coleta):
        row = coleta[coleta["ticker"] == t].iloc[0]
        print(f"  {t:8s}  status={row['status']}")

Tickers ausentes na base:            0
  Irrecuperáveis:                    0
  Ausências esperadas/parciais:      0
  Novo dado — pipeline pendente:     0

[A] Irrecuperáveis (0) — sem dados em nenhuma fonte:

[C] Só na coleta (9) — falhou mas base tem dados:
  CA        status=deletado_ticker_reciclado
  EMC       status=deletado_ticker_reciclado
  FB        status=deletado_ticker_reciclado
  KORS      status=deletado_ticker_reciclado
  LB        status=deletado_ticker_reciclado
  SE        status=deletado_ticker_reciclado
  STI       status=deletado_ticker_reciclado
  TE        status=deletado_ticker_reciclado
  VIAC      status=deletado_ticker_reciclado


In [64]:
print("=" * 60)
print("RESUMO DA BASE COMPLETA")
print("=" * 60)
print(f"\nTotal de dias:     {len(base):>6}  ({base.index[0].date()} → {base.index[-1].date()})")
print(f"Total de tickers:  {len(ticker_cols):>6}  (professor + extensão)")
print(f"Total de semestres:{base['period'].nunique():>6}  (1990/S2 → 2025/S2)")

print(f"\n--- Extensão (2016–2025) ---")
print(f"Cobertura média:    {df_cov['coverage_pct'].mean():.1f}%")
n_ausentes = len(df_aus)
n_ocorrencias = int(df_cov['n_missing'].sum())
if n_ausentes == 0:
    print(f"Tickers ausentes:   0  ✅ cobertura 100%")
else:
    print(f"Tickers ausentes:   {n_ausentes} únicos em {n_ocorrencias} ocorrências semestre-ticker")
    print(f"\n--- Tickers sem nenhum dado na extensão (irrecuperáveis) ---")
    for _, r in df_aus.iterrows():
        expected_in_period = {s['period'] for s in semesters_ext
                              for t in [r['ticker']] if t in get_constituents(s['fim'])}
        if len(expected_in_period) > 0 and r['n_sem_ausente'] == len(expected_in_period):
            print(f"  {r['ticker']:8s}  ausente em todos os {r['n_sem_ausente']} semestres esperados "
                  f"({r['primeiro_sem']} → {r['ultimo_sem']})")

df_cov.to_csv(EXTERNAL_DIR / "cobertura_extensao.csv", index=False)
df_aus.to_csv(EXTERNAL_DIR / "tickers_ausentes.csv",   index=False)
print(f"\nRelatórios salvos: cobertura_extensao.csv, tickers_ausentes.csv")

RESUMO DA BASE COMPLETA

Total de dias:       8939  (1990-07-03 → 2025-12-31)
Total de tickers:    1343  (professor + extensão)
Total de semestres:    71  (1990/S2 → 2025/S2)

--- Extensão (2016–2025) ---
Cobertura média:    100.0%
Tickers ausentes:   0  ✅ cobertura 100%

Relatórios salvos: cobertura_extensao.csv, tickers_ausentes.csv
